In [ ]:
%matplotlib ipympl

import matplotlib.pyplot as plt
import xarray as xr
import numpy as np
import colormaps as cmaps

In [ ]:
path = '/home/scratch2/meteo/microhh/rcemip-ii/output_test_ecwmf'
start_time = 172800

#path = '../rcemip_ii_case_production/experiments/budget_check'
#start_time = 0

In [ ]:
ds = dict(
    thl = xr.open_dataset(f'{path}/{start_time:07d}/3d_c/thl.zarr', consolidated=False),
    thl_tend = xr.open_dataset(f'{path}/{start_time:07d}/3d_c/thl_tend.zarr', consolidated=False),
    thl_tend_sw = xr.open_dataset(f'{path}/{start_time:07d}/3d_c/thl_tend_sw.zarr', consolidated=False),
    thl_tend_lw = xr.open_dataset(f'{path}/{start_time:07d}/3d_c/thl_tend_lw.zarr', consolidated=False),
    uthl = xr.open_dataset(f'{path}/{start_time:07d}/3d_c/uthl.zarr', consolidated=False),
    vthl = xr.open_dataset(f'{path}/{start_time:07d}/3d_c/vthl.zarr', consolidated=False),
    wthl = xr.open_dataset(f'{path}/{start_time:07d}/3d_c/wthl.zarr', consolidated=False),
    qt = xr.open_dataset(f'{path}/{start_time:07d}/3d_c/qt.zarr', consolidated=False),
    qt_tend = xr.open_dataset(f'{path}/{start_time:07d}/3d_c/qt_tend.zarr', consolidated=False),
    uqt = xr.open_dataset(f'{path}/{start_time:07d}/3d_c/uqt.zarr', consolidated=False),
    vqt = xr.open_dataset(f'{path}/{start_time:07d}/3d_c/vqt.zarr', consolidated=False),
    wqt = xr.open_dataset(f'{path}/{start_time:07d}/3d_c/wqt.zarr', consolidated=False),
)

k = 1
t = 0
cmap = plt.cm.RdBu_r

ncol = 3
nrow = 1

dx = float(ds['uthl'].x[1] - ds['uthl'].x[0])
dy = float(ds['vthl'].y[1] - ds['vthl'].y[0])
dz = float(ds['wthl'].z[k+1] - ds['wthl'].z[k])

ktot = 128
rhoref = np.fromfile(f'{path}/rhoref.0000000', dtype=np.float32)
rho = rhoref[:ktot]
rhoh = rhoref[ktot:]

z = float(ds['thl'].z[k])
print(z)

def plot_budget(var):

    fu = ds[f'u{var}'][f'u{var}']
    fv = ds[f'v{var}'][f'v{var}']
    fw = ds[f'w{var}'][f'w{var}']

    tu = -(fu[t, k, :-1, 1:   ].values - fu[t, k, :-1, :-1].values) / dx
    tv = -(fv[t, k, 1:, :-1   ].values - fv[t, k, :-1, :-1].values) / dy
    tw = -(fw[t, k+1, :-1, :-1].values - fw[t, k, :-1, :-1].values) / dz
    tw = -(rhoh[k+1]*fw[t, k+1, :-1, :-1].values - rhoh[k]*fw[t, k, :-1, :-1].values) / (rho[k] * dz)

    vmin = ds[f'{var}_tend'][f'{var}_tend'][t,k,:-1,:-1].min()
    vmax = ds[f'{var}_tend'][f'{var}_tend'][t,k,:-1,:-1].max()

    plt.figure(figsize=(10,4), layout='constrained')
    sp = 1

    ax=plt.subplot(nrow, ncol, sp); sp+=1
    plt.title(f'Total tendency {var} @ z={z} m')
    plt.imshow(ds[f'{var}_tend'][f'{var}_tend'][t,k,:-1,:-1], cmap=cmap, vmin=vmin, vmax=vmax)
    plt.colorbar()

    plt.subplot(nrow, ncol, sp, sharex=ax, sharey=ax); sp+=1
    plt.title(f'Advec tendency {var}')
    plt.imshow(tu+tv+tw, cmap=cmap, vmin=vmin, vmax=vmax)
    plt.colorbar()

    if var == 'thl':
        tr = ds['thl_tend_lw']['thl_tend_lw'][t,k,:-1,:-1] + ds['thl_tend_sw']['thl_tend_sw'][t,k,:-1,:-1]

        plt.subplot(nrow, ncol, sp, sharex=ax, sharey=ax); sp+=1
        plt.title(f'Advec + Rad tendency {var}')
        plt.imshow(tu+tv+tw+tr, cmap=cmap, vmin=vmin, vmax=vmax)
        plt.colorbar()

plot_budget('thl')
plot_budget('qt')

In [ ]:
vars_xy = dict(
        rrsg_bot          = ('000', None),
        thl_fluxbot       = ('000', None),
        qt_fluxbot        = ('000', None),
        lw_flux_dn        = ('001', (0, 128)),
        lw_flux_up        = ('001', (0, 128)),
        sw_flux_dn        = ('001', (0, 128)),
        sw_flux_up        = ('001', (0, 128)),
        sw_flux_dn_clear  = ('001', (0, 128)),
        sw_flux_up_clear  = ('001', (0, 128)),
        lw_flux_dn_clear  = ('001', (0, 128)),
        lw_flux_up_clear  = ('001', (0, 128)),
        qt_path           = ('000', None),
        qsat_path         = ('000', None),
        qlqi_path         = ('000', None),
        qi_path           = ('000', None),
        t2m               = ('000', None),
        u10m              = ('100', None),
        v10m              = ('010', None),
        thl               = ('000', (0,)),
        u                 = ('100', (0,)),
        v                 = ('010', (0,)),
        w500hpa           = ('000', None),
    )

vars_xz = dict(
        thl = '000',
        qt  = '000',
        ql  = '000',
        qi  = '000',
        qr  = '000',
        qs  = '000',
        qg  = '000',
        u   = '100',
        w   = '001',
    )

vars_dump_c = dict(
        u              = '100',
        v              = '010',
        w              = '001',
        thl            = '000',
        qt             = '000',
        ql             = '000',
        qi             = '000',
        qr             = '000',
        qs             = '000',
        qg             = '000',
        qlqi_mask      = '000',
        thl_tend       = '000',
        qt_tend        = '000',
        w_tend         = '001',
        thl_tend_lw    = '000',
        thl_tend_sw    = '000',
        qrsg_tend_sed  = '000',
        qtr_tend_frz   = '000',
        uthl           = '100',
        vthl           = '010',
        wthl           = '001',
        uqt            = '100',
        vqt            = '010',
        wqt            = '001',
        wqr            = '001',
        wql            = '001',
        wqi            = '001',
        uw             = '101',
        vw             = '011',
    )

In [ ]:
t = -1
k = 1
cmap = cmaps.WhiteBlueGreenYellowRed

N = int(np.ceil(len(vars_xy)**0.5))

for coarse in [False]:

    plt.figure(figsize=(14, 12), layout='constrained')
    sp = 1

    for name,loc in vars_xy.items():
        print(name)
        folder = 'xy_c' if coarse else 'xy'

        if loc[1] is None:
            ds = xr.open_dataset(f'{path}/{start_time:07d}/{folder}/{name}.zarr', consolidated=False)
        else:
            ds = xr.open_dataset(f'{path}/{start_time:07d}/{folder}/{name}_{loc[1][0]}.zarr', consolidated=False)
            name += f'_{loc[1][0]}'

        var = ds[name][t,:,:]
        vmin = float(var.min())
        vmax = float(var.max())

        plt.subplot(N, N, sp); sp+=1
        plt.title(f'{name}: range=({vmin:.1e} - {vmax:.1e})', loc='left', size=8)
        plt.imshow(var, cmap=cmap, aspect='auto')
        plt.colorbar()

        plt.savefig(f'{folder}_all.png')

In [ ]:
N = int(np.ceil(len(vars_xz)**0.5))

plt.figure(figsize=(8,8), layout='constrained')
sp = 1

for name,loc in vars_xz.items():
    name = f'{name}_ymean'
    ds = xr.open_dataset(f'{path}/{start_time:07d}/xz/{name}.zarr', consolidated=False)

    var = ds[name][t,:,:]
    vmin = float(var.min())
    vmax = float(var.max())

    plt.subplot(N, N, sp); sp+=1
    plt.title(f'{name}: range=({vmin:.1e} - {vmax:.1e})', loc='left', size=8)
    plt.imshow(var, cmap=cmap, aspect='auto', origin='lower')
    plt.colorbar()

plt.savefig('xz_all.png')

In [ ]:
N = int(np.ceil(len(vars_dump_c)**0.5))
k = 1

plt.figure(figsize=(12,12), layout='constrained')
sp = 1

for name,loc in vars_dump_c.items():
    ds = xr.open_dataset(f'{path}/{start_time:07d}/3d_c/{name}.zarr', consolidated=False)

    var = ds[name][t,k,:,:]
    vmin = float(var.min())
    vmax = float(var.max())

    plt.subplot(N, N, sp); sp+=1
    plt.title(f'{name}: range=({vmin:.1e} - {vmax:.1e})', loc='left', size=8)
    plt.imshow(var, cmap=cmap, aspect='auto', origin='lower')
    plt.colorbar()

plt.savefig('dump_all.png')